In [1]:
import pandas as pd

In [7]:
files = {
    'GE': 'postagging_genoese_ORDINATO.csv',
    'IT': 'postagging_italian.csv',
    'FR': 'postagging_french.csv',
    'PT': 'postagging_portuguese.csv',
}

dfs = []
for lang, file in files.items():
	df = pd.read_csv(file)
	rename_cols = {'POS_GROUPED': f'POS_{lang}'}
	df = df.rename(columns=rename_cols)
	df = df[[c for c in df.columns if c in ['SENTENCE', 'TEXT', f'POS_{lang}']]]
	df['word_instance'] = df.groupby('TEXT').cumcount()

	if lang == 'GE':
		df['original_index'] = range(len(df))  # aggiungi una colonna per mantenere l'ordine originale
	else:
		df = df.drop(columns=['SENTENCE'])  # rimuovi la colonna SENTENCE per gli altri file

	dfs.append(df)
    
merged_df = dfs[0]
for df in dfs[1:]:
	merged_df = pd.merge(merged_df, df, on=['TEXT', 'word_instance'], how='outer')
drop_cols = [cols for cols in merged_df.columns if '_drop' in cols]
merged_df = merged_df.drop(columns=drop_cols)

if 'original_index' in merged_df.columns:
    merged_df = merged_df.sort_values('original_index')
    merged_df = merged_df.drop(columns=['original_index'])

merged_df = merged_df.drop(columns=['word_instance']) # rimozione colonna tecnica
merged_df.to_csv('postagging_merged.csv', index=False)
merged_df.head()

,SENTENCE,TEXT,POS_GE,POS_IT,POS_FR,POS_PT
803,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",O,PRON,NOUN,NOUN,PRON
1018,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",Zouhaur,NOUN,NOUN,NOUN,NOUN
541,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",Atif,NOUN,NOUN,NOUN,NOUN
3,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",",",PUNCT,PUNCT,PUNCT,PUNCT
3248,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",o,PRON,FUNC,FUNC,PRON
